# TAIL-FLY Phase A — ImageNet-R train-only development

Run every cell in order on a Colab T4 GPU. This notebook uses only ImageNet-R training data for model selection, keeps `test.pt` absent, and prints short `WTA CACHE`, `START`, `TASK`, `DONE`, and `RESUME` progress lines. A pass is only a development gate; it is not a paper result.

In [ ]:
# === Edit paths/source only. Do not edit seed, grid, model, or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/tail-fly'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
DRIVE_WTA_CACHE = f'{DRIVE_ROOT}/tail_fly_imagenetr_wta_cache_seed2025'
WTA_CACHE_DIR = '/content/tail_fly_imagenetr_wta_cache_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/tail_fly_imagenetr_phasea_seed2025'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
SEED = 2025
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CONFIG_SHA256 = 'c49b6a7e813c94d40413dd2d4f8e5e7889fff9c5b1aea0a5e7af046c0913bc04'

In [ ]:
# Runtime setup. chdir first so deleting an old clone cannot invalidate cwd.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists(): shutil.rmtree(repo_path)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, f'Clone failed ({clone.returncode}). Confirm the branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/tail_fly_imagenetr_train_only.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked config identity mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked seed:', SEED, '| config SHA-256:', CONFIG_SHA256)

In [ ]:
# Obtain and verify the exact frozen ViT checkpoint.
if CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', 'model.safetensors')
elif CHECKPOINT_SOURCE == 'google_drive':
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
else:
    raise ValueError('CHECKPOINT_SOURCE must be huggingface or google_drive')
checkpoint = Path(CHECKPOINT_PATH)
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
assert checkpoint.stat().st_size == CHECKPOINT_SIZE, 'Checkpoint size mismatch.'
assert digest == CHECKPOINT_SHA256, 'Checkpoint SHA-256 mismatch.'
print('checkpoint PASS:', checkpoint)
print('SHA-256:', digest)

In [ ]:
# Resolve the processed ImageNet-R Kaggle artifact. This indexes test paths but does not extract test features.
import kagglehub
from torchvision.datasets import ImageFolder
download_root = Path(kagglehub.dataset_download('zaphat206/imagenet-r')).resolve()
directories = [download_root] + [p for p in download_root.rglob('*') if p.is_dir()]
matches = sorted({p.resolve() for p in directories if (p/'train').is_dir() and (p/'test').is_dir()})
assert len(matches) == 1, f'Expected one processed root, found: {matches}'
image_root = matches[0]
processed_root = Path('/content/processed_datasets')
loader_link = processed_root / 'imagenet-r'
processed_root.mkdir(parents=True, exist_ok=True)
if loader_link.is_symlink() or loader_link.is_file(): loader_link.unlink()
elif loader_link.exists(): shutil.rmtree(loader_link)
loader_link.symlink_to(image_root, target_is_directory=True)
train_index, test_index = ImageFolder(image_root/'train'), ImageFolder(image_root/'test')
assert len(train_index.classes) == len(test_index.classes) == 200
assert train_index.class_to_idx == test_index.class_to_idx
print('dataset root:', image_root)
print('train/test samples:', len(train_index), len(test_index), '| classes:', len(train_index.classes))

In [ ]:
# A0/A1 correctness tests; these use synthetic data only.
tests = ['tests/test_tail_fly_math.py', 'tests/test_tail_fly_learner.py', 'tests/test_tail_fly_phasea.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('TAIL-FLY correctness gate: PASS')

In [ ]:
# Restore or extract TRAIN embeddings only; progress is printed per task.
local_cache, drive_cache = Path(TRAIN_CACHE_DIR), Path(DRIVE_TRAIN_CACHE)
if not (local_cache/'metadata.json').is_file():
    if (drive_cache/'metadata.json').is_file():
        print('Restoring train-only feature cache from Drive...', flush=True)
        local_cache.mkdir(parents=True, exist_ok=True)
        for source in sorted(drive_cache.iterdir()):
            print('COPY', source.name, flush=True); shutil.copy2(source, local_cache/source.name)
    else:
        command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--extract-train-only', '--root', str(processed_root), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', TRAIN_CACHE_DIR, '--output-dir', '/content/tail_fly_imagenetr_extract', '--dataset', 'ImageNet-R', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '200', '--num-tasks', '20', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
        print('Starting 20-task TRAIN-only ViT extraction...', flush=True)
        subprocess.run(command, check=True)
        assert not (local_cache/'test.pt').exists()
        assert not drive_cache.exists(), 'Incomplete Drive feature cache exists; inspect it first.'
        drive_cache.mkdir(parents=True)
        for source in sorted(local_cache.iterdir()):
            print('SAVE', source.name, 'to Drive', flush=True); shutil.copy2(source, drive_cache/source.name)
metadata = json.loads((local_cache/'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False and not (local_cache/'test.pt').exists()
print('train feature cache PASS:', metadata['train_shape'], '| test.pt absent')

In [ ]:
# Restore the large WTA infrastructure cache when available. Missing cache is built by the next cell with live progress.
local_wta, drive_wta = Path(WTA_CACHE_DIR), Path(DRIVE_WTA_CACHE)
if not (local_wta/'metadata.json').is_file() and (drive_wta/'metadata.json').is_file():
    print('Restoring WTA cache from Drive (large copy)...', flush=True)
    local_wta.mkdir(parents=True, exist_ok=True)
    for source in sorted(drive_wta.iterdir()):
        print('COPY', source.name, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
        shutil.copy2(source, local_wta/source.name)
print('WTA cache status:', 'restored' if (local_wta/'metadata.json').is_file() else 'will be created')

In [ ]:
# Locked train-only run. Completed exact/raw/rank units resume from Drive after interruption.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path/'locked_config.json')
command = [sys.executable, '-u', 'tools/tail_fly_phasea.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--code-cache-dir', WTA_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting TAIL-FLY Phase A.', flush=True)
print('Progress legend: WTA CACHE=create/verify; START/DONE=unit; TASK=stage; RESUME=completed unit.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'Runner elapsed: {(time.time()-started)/60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'Runner failed; send the full traceback without editing config.'
assert (output_path/'phasea_results.json').is_file() and not (local_cache/'test.pt').exists()
print('TAIL-FLY Phase A process: COMPLETE')

In [ ]:
# Persist a newly created WTA cache, show compact results, download evidence, then STOP.
if (local_wta/'metadata.json').is_file() and not (drive_wta/'metadata.json').is_file():
    assert not drive_wta.exists(), 'Incomplete Drive WTA cache exists; inspect it first.'
    drive_wta.mkdir(parents=True)
    for source in sorted(local_wta.iterdir()):
        print('SAVE', source.name, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
        shutil.copy2(source, drive_wta/source.name)
import pandas as pd
result = json.loads((output_path/'phasea_results.json').read_text())
rows = [{k: row.get(k) for k in ('method','rank','ridge_lambda','validation_average_accuracy','persistent_state_bytes')} for row in result['candidates']]
display(pd.DataFrame(rows).sort_values(['method','validation_average_accuracy'], ascending=[True,False]))
print('selected:', json.dumps(result['selected_tail_config'], indent=2))
print('decision:', result['decision'])
print('gates:', json.dumps(result['gates'], indent=2))
print('diagnostics:', json.dumps(result['gate_diagnostics'], indent=2))
archive = shutil.make_archive('/content/tail_fly_imagenetr_phasea_train_only', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP for audit; do not evaluate ImageNet-R test.')